# IA Generativa, Foundation Models, RAG y AWS Bedrock

Este notebook resume y representa los temas que hemos venido trabajando:

1. Diferencia entre IA tradicional e IA generativa  
2. Foundation Models, pre-entrenamiento, adaptación y despliegue  
3. Transformers y LLMs  
4. Por qué los LLMs necesitan modelos grandes  
5. Cómo responder con información actualizada usando RAG  
6. Cómo construir un chatbot empresarial con Amazon Bedrock, Knowledge Bases, S3 y Guardrails  
7. Caso práctico: asistente virtual para pólizas de seguros almacenadas en PDF  

> Objetivo: entender el flujo conceptual y técnico sin necesidad de entrenar un modelo desde cero.


## 1. IA tradicional vs IA generativa

La **IA tradicional** normalmente aprende a tomar decisiones sobre datos existentes.

Ejemplo:

- Entrada: texto de un email
- Salida: `spam` o `no spam`

La **IA generativa** aprende a producir contenido nuevo.

Ejemplo:

- Entrada: “Escribe un email profesional para un cliente”
- Salida: un email completo, coherente y adaptado al contexto.

La gran diferencia es el tipo de salida:

| Tipo de IA | Entrada | Salida | Ejemplo |
|---|---|---|---|
| IA tradicional | Datos | Etiqueta o predicción | Spam / No spam |
| IA generativa | Instrucción o contexto | Texto, imagen, código, audio | Email completo |


In [ ]:
# Ejemplo conceptual de IA tradicional: clasificar un email como spam o no spam

email = "Gana dinero rapido con esta oferta exclusiva"

palabras_spam = ["gana", "dinero", "oferta", "gratis", "exclusiva"]

score_spam = 0

for palabra in palabras_spam:
    if palabra in email.lower():
        score_spam += 1

if score_spam >= 2:
    resultado = "spam"
else:
    resultado = "no spam"

print("Email:", email)
print("Puntaje de spam:", score_spam)
print("Clasificacion:", resultado)


### Explicación detallada del código anterior

- `email = ...`  
  Guarda el texto del email que queremos analizar.

- `palabras_spam = [...]`  
  Define una lista simple de palabras que suelen aparecer en correos sospechosos.

- `score_spam = 0`  
  Inicializa un contador. Este contador aumenta cada vez que encontramos una palabra sospechosa.

- `for palabra in palabras_spam:`  
  Recorre una por una las palabras de la lista.

- `if palabra in email.lower():`  
  Convierte el email a minúsculas y verifica si contiene esa palabra.

- `score_spam += 1`  
  Si la palabra aparece, aumenta el puntaje de spam.

- `if score_spam >= 2:`  
  Si aparecen dos o más palabras sospechosas, clasificamos el email como spam.

Este ejemplo es muy simple, pero representa la idea central de la IA tradicional: **clasificar**.


In [ ]:
# Ejemplo conceptual de IA generativa: construir un email usando una plantilla

cliente = "Laura"
producto = "poliza de seguro de vida"
tono = "profesional y claro"

email_generado = f"""
Hola {cliente},

Gracias por tu interes en nuestra {producto}.

Queremos informarte que nuestro equipo esta disponible para resolver tus dudas
y ayudarte a elegir la cobertura que mejor se adapte a tus necesidades.

Quedamos atentos a tus comentarios.

Cordialmente,
Equipo de Atencion al Cliente
"""

print(email_generado)


### Explicación detallada del código anterior

Este ejemplo no usa un LLM real, pero sirve para visualizar la diferencia.

- `cliente = "Laura"`  
  Variable con el nombre de la persona.

- `producto = "poliza de seguro de vida"`  
  Variable con el producto sobre el que se escribirá.

- `tono = "profesional y claro"`  
  Indica el estilo deseado. En este ejemplo no se usa directamente, pero en un LLM real sí sería parte del prompt.

- `email_generado = f""" ... """`  
  Crea un texto completo usando una cadena multilínea.

- `{cliente}` y `{producto}`  
  Insertan valores dentro del texto.

Este ejemplo representa la idea de la IA generativa: **producir contenido nuevo**, no solo clasificar.


## 2. ¿Por qué la IA generativa necesita modelos más grandes?

Un clasificador de spam solo necesita elegir entre pocas opciones.

En cambio, un modelo generativo debe decidir la siguiente palabra muchas veces:

> “Hola Laura, gracias por...”

En cada posición debe elegir entre miles de posibles tokens. Además debe mantener:

- Gramática
- Coherencia
- Tono
- Contexto
- Conocimiento del mundo
- Relación entre frases largas

Por eso necesita muchos más parámetros.


In [ ]:
# Simulacion conceptual: elegir la siguiente palabra segun probabilidades

import random

opciones_siguiente_palabra = {
    "gracias": 0.40,
    "lamentamos": 0.15,
    "confirmamos": 0.25,
    "recordamos": 0.20
}

palabras = list(opciones_siguiente_palabra.keys())
probabilidades = list(opciones_siguiente_palabra.values())

siguiente_palabra = random.choices(
    palabras,
    weights=probabilidades,
    k=1
)[0]

print("Siguiente palabra generada:", siguiente_palabra)


### Explicación detallada del código anterior

- `import random`  
  Importa una librería estándar de Python para generar selecciones aleatorias.

- `opciones_siguiente_palabra = {...}`  
  Define posibles palabras que podrían venir después en una frase, junto con una probabilidad conceptual.

- `palabras = list(...)`  
  Extrae solo las palabras del diccionario.

- `probabilidades = list(...)`  
  Extrae los pesos o probabilidades.

- `random.choices(...)`  
  Escoge una palabra teniendo en cuenta los pesos.

- `weights=probabilidades`  
  Hace que palabras con mayor probabilidad sean más fáciles de escoger.

- `k=1`  
  Indica que queremos una sola palabra.

Un LLM real hace algo mucho más complejo, pero conceptualmente predice tokens de forma secuencial.


## 3. Foundation Models

Un **Foundation Model** es un modelo grande entrenado con enormes cantidades de datos generales.

Puede servir como base para muchas tareas:

- Redacción
- Resumen
- Traducción
- Preguntas y respuestas
- Código
- Análisis de documentos

Un **LLM** es un Foundation Model especializado en lenguaje.


## 4. Ciclo de vida: pre-entrenamiento, adaptación y despliegue

El ciclo típico es:

1. **Pre-entrenamiento**  
   El modelo aprende patrones generales usando grandes volúmenes de datos.

2. **Adaptación**  
   El modelo se ajusta para seguir instrucciones, adoptar un estilo o especializarse en una tarea.

3. **Despliegue**  
   El modelo se pone en producción mediante una API, chatbot o aplicación.


In [ ]:
# Representacion simple del ciclo de vida de un foundation model

etapas = [
    "1. Pre-entrenamiento con datos masivos",
    "2. Adaptacion o alineacion para tareas especificas",
    "3. Despliegue en una aplicacion o API",
    "4. Monitoreo y mejora continua"
]

for etapa in etapas:
    print(etapa)


### Explicación detallada del código anterior

- `etapas = [...]`  
  Crea una lista con las fases principales del ciclo de vida.

- `for etapa in etapas:`  
  Recorre cada elemento de la lista.

- `print(etapa)`  
  Muestra cada fase en pantalla.

Este bloque no entrena un modelo; solo representa el proceso conceptual.


## 5. Transformers y LLMs

El paper **“Attention is All You Need” (2017)** introdujo la arquitectura Transformer.

La idea central es la **self-attention**:

> Cada token puede mirar otros tokens del contexto para decidir qué información es importante.

Esto permitió:

- Mejor paralelización
- Mejor manejo de contexto largo
- Modelos más escalables
- Base para modelos como GPT, Claude y Llama


In [ ]:
# Simulacion conceptual de atencion: palabras relevantes para entender una frase

frase = ["La", "poliza", "cubre", "accidentes", "en", "viajes"]

pesos_atencion = {
    "La": 0.05,
    "poliza": 0.35,
    "cubre": 0.25,
    "accidentes": 0.25,
    "en": 0.03,
    "viajes": 0.07
}

for palabra, peso in pesos_atencion.items():
    print(f"{palabra}: importancia {peso}")


### Explicación detallada del código anterior

- `frase = [...]`  
  Representa una frase dividida en palabras o tokens.

- `pesos_atencion = {...}`  
  Asigna un peso conceptual a cada palabra.

- `"poliza": 0.35`  
  Indica que “poliza” es muy relevante para entender la frase.

- `for palabra, peso in pesos_atencion.items():`  
  Recorre cada palabra y su peso asociado.

- `print(...)`  
  Muestra la importancia de cada palabra.

En un Transformer real, estos pesos se calculan matemáticamente usando matrices llamadas Query, Key y Value.


## 6. Problema: modelo entrenado hasta 2023

Si un modelo fue entrenado con datos hasta 2023, no conoce de forma interna eventos de 2024 o posteriores.

Puede:

- Inferir
- Adivinar
- Generalizar

Pero eso no garantiza precisión.

Para darle información actualizada sin re-entrenarlo se usa normalmente **RAG**.


## 7. RAG: Retrieval-Augmented Generation

RAG significa **Retrieval-Augmented Generation**.

El flujo es:

1. El usuario hace una pregunta.
2. El sistema busca documentos relevantes.
3. Se recuperan fragmentos de información.
4. El LLM responde usando esos fragmentos como contexto.
5. Si no hay evidencia, debe responder “no tengo información suficiente”.

Esto reduce alucinaciones y evita reentrenar el modelo.


In [ ]:
# Simulacion simple de RAG con documentos internos

documentos = [
    {
        "id": 1,
        "texto": "La poliza de vida cubre fallecimiento por causas naturales y accidentales."
    },
    {
        "id": 2,
        "texto": "La poliza de viaje cubre perdida de equipaje y asistencia medica internacional."
    },
    {
        "id": 3,
        "texto": "La poliza de auto cubre danos a terceros y perdida total por accidente."
    }
]

pregunta = "Que cubre la poliza de viaje?"

resultados = []

for doc in documentos:
    if "viaje" in doc["texto"].lower():
        resultados.append(doc)

print("Pregunta:", pregunta)
print("\nDocumentos recuperados:")

for resultado in resultados:
    print("-", resultado["texto"])


### Explicación detallada del código anterior

- `documentos = [...]`  
  Simula una pequeña base documental interna.

- Cada documento tiene:
  - `id`: identificador
  - `texto`: contenido

- `pregunta = ...`  
  Representa la pregunta del usuario.

- `resultados = []`  
  Lista vacía donde guardaremos documentos relevantes.

- `for doc in documentos:`  
  Recorre los documentos disponibles.

- `if "viaje" in doc["texto"].lower():`  
  Busca documentos que contengan la palabra “viaje”.

- `resultados.append(doc)`  
  Agrega el documento relevante a los resultados.

Esto simula la parte de **retrieval** de RAG. En producción se usan embeddings y bases vectoriales, no una búsqueda tan simple.


In [ ]:
# Simulacion de respuesta fundamentada usando los documentos recuperados

if resultados:
    contexto = resultados[0]["texto"]
    respuesta = f"Segun el documento recuperado: {contexto}"
else:
    respuesta = "No tengo informacion suficiente en los documentos disponibles para responder."

print(respuesta)


### Explicación detallada del código anterior

- `if resultados:`  
  Verifica si se encontró al menos un documento relevante.

- `contexto = resultados[0]["texto"]`  
  Toma el texto del primer documento recuperado.

- `respuesta = f"...{contexto}"`  
  Construye una respuesta usando únicamente el documento.

- `else:`  
  Si no se recuperó ningún documento, evita inventar.

- `No tengo informacion suficiente...`  
  Esta es una práctica clave para sistemas empresariales que requieren precisión.

Este patrón es fundamental: **si no hay evidencia, no se responde inventando**.


## 8. ¿Fine-tuning, RAG o prompt engineering?

Para responder solo con datos de la empresa, la mejor estrategia suele ser:

## RAG + Prompt Engineering

| Estrategia | Ventaja | Problema |
|---|---|---|
| Fine-tuning | Ajusta estilo o comportamiento | No garantiza datos actualizados |
| Prompt engineering | Barato y rápido | No da acceso real a documentos |
| RAG | Usa documentos reales | Requiere indexar y recuperar bien |

Conclusión:

> RAG combina mejor precisión y costo cuando la fuente de verdad son documentos empresariales.


## 9. Amazon Bedrock

**Amazon Bedrock** es un servicio administrado de AWS para construir aplicaciones de IA generativa usando Foundation Models.

Permite:

- Acceder a modelos como Claude, Llama, Titan, entre otros.
- Crear aplicaciones sin administrar infraestructura.
- Integrar RAG con Knowledge Bases.
- Usar Guardrails para seguridad y control.


## 10. Arquitectura recomendada para PDFs de pólizas en S3

Caso:

> Una empresa de seguros quiere un asistente que responda preguntas usando pólizas vigentes en PDF almacenadas en S3. No quiere inventos y no tiene experiencia en ML.

Solución recomendada:

1. **Amazon S3**  
   Almacena los PDFs de pólizas.

2. **Amazon Bedrock Knowledge Bases**  
   Indexa los documentos, crea embeddings y permite recuperación semántica.

3. **Vector Store**  
   Puede ser Amazon OpenSearch Serverless u otra base vectorial compatible.

4. **Amazon Bedrock**  
   Ejecuta el Foundation Model que genera la respuesta.

5. **Amazon Bedrock Guardrails**  
   Controla contenido inapropiado y ayuda a limitar respuestas no fundamentadas.

6. **Aplicación web o chatbot**  
   Interfaz donde el cliente hace preguntas.


In [ ]:
# Representacion de arquitectura como pasos de procesamiento

arquitectura = {
    "1_usuario": "Cliente hace una pregunta sobre su poliza",
    "2_app": "La aplicacion recibe la pregunta",
    "3_knowledge_base": "Bedrock Knowledge Bases busca fragmentos relevantes en los PDFs",
    "4_modelo": "Amazon Bedrock genera una respuesta usando el contexto recuperado",
    "5_guardrails": "Guardrails valida seguridad, tono y fundamentacion",
    "6_respuesta": "El cliente recibe una respuesta basada en documentos"
}

for componente, descripcion in arquitectura.items():
    print(componente, "->", descripcion)


### Explicación detallada del código anterior

- `arquitectura = {...}`  
  Crea un diccionario que representa los componentes principales del sistema.

- Las claves como `"1_usuario"` o `"3_knowledge_base"` indican el orden lógico del flujo.

- Los valores describen qué ocurre en cada paso.

- `for componente, descripcion in arquitectura.items():`  
  Recorre cada componente del diccionario.

- `print(componente, "->", descripcion)`  
  Muestra el flujo completo de la arquitectura.

Este código representa el recorrido de una pregunta desde el usuario hasta la respuesta final.


## 11. Por qué la opción correcta era Bedrock con Knowledge Bases y Guardrails

En el caso de selección múltiple, la mejor respuesta era:

## B. Usar Amazon Bedrock con Knowledge Bases para indexar PDFs y Guardrails para controlar respuestas.

Razones:

- No requiere experiencia avanzada en Machine Learning.
- Evita entrenar un modelo desde cero.
- Usa los PDFs en S3 como fuente de verdad.
- Knowledge Bases implementa el patrón RAG.
- Guardrails ayuda a controlar contenido inapropiado y respuestas no fundamentadas.
- Es más rápido y costo-eficiente que entrenar un modelo personalizado.


In [ ]:
# Comparacion conceptual de opciones

opciones = {
    "A_SageMaker": {
        "requiere_ml": True,
        "costo": "alto",
        "adecuada": False
    },
    "B_Bedrock_KnowledgeBases_Guardrails": {
        "requiere_ml": False,
        "costo": "medio/bajo",
        "adecuada": True
    },
    "C_Lex": {
        "requiere_ml": False,
        "costo": "medio",
        "adecuada": False
    },
    "D_Comprehend_Kendra": {
        "requiere_ml": False,
        "costo": "medio",
        "adecuada": "parcial"
    }
}

for opcion, detalle in opciones.items():
    print("\n", opcion)
    for clave, valor in detalle.items():
        print(" ", clave, ":", valor)


### Explicación detallada del código anterior

- `opciones = {...}`  
  Crea un diccionario con las alternativas del caso.

- Cada alternativa tiene atributos:
  - `requiere_ml`: si necesita experiencia en Machine Learning.
  - `costo`: estimación conceptual del costo.
  - `adecuada`: si cumple bien el caso.

- `for opcion, detalle in opciones.items():`  
  Recorre cada opción.

- `for clave, valor in detalle.items():`  
  Recorre los atributos internos de cada opción.

Este bloque ayuda a visualizar por qué Bedrock + Knowledge Bases + Guardrails es la solución más adecuada.


## 12. Prompt recomendado para evitar respuestas inventadas

Un prompt empresarial para RAG podría decir:

```text
Eres un asistente de seguros.
Responde únicamente usando el contexto recuperado de las pólizas vigentes.
Si la respuesta no aparece explícitamente en los documentos, responde:
"No tengo información suficiente en las pólizas disponibles para responder."
No inventes coberturas, exclusiones, fechas, precios ni condiciones.
Incluye la fuente o fragmento usado cuando esté disponible.
```

Este prompt no reemplaza RAG, pero ayuda a controlar el comportamiento del modelo.


## 13. Conclusión general

- La IA tradicional clasifica o predice.
- La IA generativa crea contenido.
- Los Foundation Models son modelos base entrenados a gran escala.
- Los LLMs son Foundation Models especializados en lenguaje.
- Los Transformers hicieron posible el escalamiento moderno de los LLMs.
- Si el modelo no tiene información actualizada, se usa RAG.
- Para empresas con documentos internos, RAG suele ser mejor que fine-tuning.
- En AWS, la solución administrada más adecuada es Amazon Bedrock con Knowledge Bases y Guardrails.
